# 03 — Visualize the model (inputs → architecture → outputs)

Four views of the **exact** `FloodConvLSTM` defined in
[`02_train_model.ipynb`](02_train_model.ipynb):

1. **`torchinfo` summary** — every layer with its input/output shape and params.
2. **Architecture diagram** — the whole pipeline as labelled blocks with shapes.
3. **A real sample, end to end** — the actual input tensors and the model's output.
4. **Node graph (`torchview`)** — the network's computation graph.

> Read-only / no training. We rebuild the model from notebook 02 so this always
> matches it. A forward pass runs at full resolution on one GPU.

## 1. Rebuild the model from notebook 02

We exec notebook 02's code cells (config, pooling index, dataset, model, LightningModule) — skipping the training / plotting / test cells — so the model here is identical to it.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

nb2 = json.loads((ROOT / "notebooks" / "model" / "02_train_model.ipynb").read_text())
_SKIP = ("trainer = L.Trainer", "mpath =", "tester.predict", "best_gpu")
for _c in nb2["cells"]:
    if _c["cell_type"] != "code":
        continue
    _src = "".join(_c["source"])
    if any(s in _src for s in _SKIP):
        continue
    exec(_src, globals())

import torch

DEV = "cuda:0"
net = FloodConvLSTM().to(DEV).eval()           # the network (no Lightning wrapper)
lm = FloodNet()                                # LightningModule (loss/metric logic)
n_par = sum(p.numel() for p in net.parameters())
print(f"\nmodel rebuilt | {n_par:,} parameters")
print(f"input  : (B, T={T_FRAMES}, C={N_CH}, {IMG_H}, {IMG_W})")
print(f"feature: encoder /{POOL_STRIDE} -> ({S_H}, {S_W}) -> ConvLSTM")
print(f"output : (B, {GRID_R}, {GRID_C})  on the {CELL_KM} km grid")

## 2. `torchinfo` summary — layers, shapes, parameters

Runs one forward pass and reports each module's input/output shape and parameter
count. This is the most detailed textual view of the network.

In [ ]:
from torchinfo import summary

summary(net, input_size=(1, T_FRAMES, N_CH, IMG_H, IMG_W), device=DEV, depth=3,
        col_names=("input_size", "output_size", "num_params"),
        row_settings=("var_names",))

## 3. Architecture diagram

The full data flow as labelled blocks, with the tensor shape and parameter count
at each stage — inputs on the left, the 50 km flood map on the right.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow, FancyBboxPatch


def _p(m):
    return sum(q.numel() for q in m.parameters())


enc_p = sum(_p(b) for b in net.blocks)
stages = [
    ("INPUT\nGOES + GLM", f"(T={T_FRAMES}, {N_CH},\n{IMG_H}x{IMG_W})", "", "#cfe8ff"),
    ("Encoder\n3x conv-block\n+ max-pool", f"(T={T_FRAMES}, 64,\n{S_H}x{S_W})",
     f"{enc_p/1e3:.0f}K", "#d9f2d0"),
    ("ConvLSTM\n2 layers\n(fuse 6 frames)", f"(64,\n{S_H}x{S_W})",
     f"{_p(net.lstm)/1e3:.0f}K", "#d9f2d0"),
    ("CellPool\nGOES px -> 50km\n(scatter-mean)", f"(64,\n{GRID_R}x{GRID_C})",
     "0", "#ffe6b3"),
    ("Grid head\nconv-block + 1x1", f"(1,\n{GRID_R}x{GRID_C})",
     f"{_p(net.head)/1e3:.0f}K", "#d9f2d0"),
    ("OUTPUT\nflood logits", f"({GRID_R}x{GRID_C})", "", "#ffd0d0"),
]

fig, ax = plt.subplots(figsize=(15, 3.2))
x = 0.0
for i, (name, shape, params, color) in enumerate(stages):
    ax.add_patch(FancyBboxPatch((x, 0.25), 1.7, 1.5, boxstyle="round,pad=0.04",
                                fc=color, ec="#333"))
    ax.text(x + 0.85, 1.35, name, ha="center", va="center", fontsize=9,
            weight="bold")
    ax.text(x + 0.85, 0.72, shape, ha="center", va="center", fontsize=8,
            family="monospace")
    if params:
        ax.text(x + 0.85, 0.38, f"{params} params", ha="center", va="center",
                fontsize=7, style="italic", color="#555")
    if i < len(stages) - 1:
        ax.add_patch(FancyArrow(x + 1.72, 1.0, 0.33, 0, width=0.03,
                                head_width=0.16, head_length=0.12, fc="#333"))
    x += 2.1
ax.set_xlim(-0.1, x); ax.set_ylim(0, 2.0); ax.axis("off")
ax.set_title(f"FloodConvLSTM  —  {n_par:,} parameters  "
             f"(POOL_STRIDE={POOL_STRIDE}, CELL_KM={CELL_KM})", fontsize=11)
plt.tight_layout()

## 4. A real sample, end to end

The **actual inputs** (one band across the 6 frames + the whole-day lightning
map) and the **outputs** (model probability, the neighborhood-blurred training
target, and the hard label) for one training day.

In [ ]:
import numpy as np

x, y = ds_train[0]                              # x (6,7,H,W) f16, y (59,95)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    prob = torch.sigmoid(net(x[None].to(DEV)))[0].float().cpu().numpy()
soft = lm._soft_target(y[None])[0].numpy()     # neighborhood-blurred target

fig, ax = plt.subplots(2, 4, figsize=(15, 6))
for t in range(4):                             # inputs: band 13 at 4 of 6 frames
    ax[0, t].imshow(x[t, 4, ::6, ::6], cmap="gray_r")
    ax[0, t].set_title(f"input: band13  frame {t}", fontsize=9)
    ax[0, t].axis("off")

ax[1, 0].imshow(x[3, 6, ::6, ::6], cmap="magma")     # GLM channel
ax[1, 0].set_title("input: GLM whole-day", fontsize=9); ax[1, 0].axis("off")
mp = lambda a: np.where(land_mask, a, np.nan)
im = ax[1, 1].imshow(mp(prob), cmap="viridis", vmin=0, vmax=1)
ax[1, 1].set_title("output: P(flood)", fontsize=9); ax[1, 1].axis("off")
ax[1, 2].imshow(mp(soft), cmap="Oranges", vmin=0, vmax=1)
ax[1, 2].set_title("train target (blurred)", fontsize=9); ax[1, 2].axis("off")
ax[1, 3].imshow(mp(y.numpy()), cmap="Reds", vmin=0, vmax=1)
ax[1, 3].set_title(f"hard label ({int(y.sum())} cells)", fontsize=9)
ax[1, 3].axis("off")
fig.colorbar(im, ax=ax[1, 1], shrink=0.7)
plt.tight_layout()

## 5. Network node graph (`torchview`)

`torchview` traces a forward pass and builds the computation graph. `dot`
(graphviz binary) isn't installed here, so we save the graph's **DOT source** —
view it by pasting into an online Graphviz viewer
([GraphvizOnline](https://dreampuf.github.io/GraphvizOnline/)) or, once graphviz
is installed, `dot -Tpng floodnet_graph.gv -o graph.png`. If `dot` is on PATH we
also render it inline below.

In [ ]:
import shutil

from torchview import draw_graph

CKPT_DIR.mkdir(parents=True, exist_ok=True)
graph = draw_graph(net, input_size=(1, T_FRAMES, N_CH, IMG_H, IMG_W), device=DEV,
                   depth=2, expand_nested=False, save_graph=False)
gv_path = CKPT_DIR / "floodnet_graph.gv"
gv_path.write_text(graph.visual_graph.source)
print(f"saved DOT graph -> {gv_path}  ({len(graph.visual_graph.source):,} chars)")
print("view: paste into https://dreampuf.github.io/GraphvizOnline/")

if shutil.which("dot"):                        # render inline if graphviz present
    graph.visual_graph.render(str(CKPT_DIR / "floodnet_graph"), format="png",
                              cleanup=True)
    from IPython.display import Image, display
    display(Image(str(CKPT_DIR / "floodnet_graph.png")))
else:
    print("(no 'dot' on PATH -> inline image skipped; use the .gv as above)")